In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32).view(-1, 1)




In [ ]:
# 2. Create TensorDataset objects
train_ds = TensorDataset(X_train_t, y_train_t)
test_ds  = TensorDataset(X_test_t,  y_test_t)




In [ ]:
# 3. Create DataLoaders
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)



In [ ]:
# 4. Print shape of one batch
xb, yb = next(iter(train_loader))
print("X batch:", xb.shape)
print("y batch:", yb.shape)



In [ ]:
# 5. Display sample images
xb, yb = next(iter(train_loader))

plt.figure(figsize=(10, 4))
for i in range(6):
    img = xb[i].permute(1, 2, 0).numpy()  # (C,H,W) -> (H,W,C)
    plt.subplot(2, 3, i+1)
    plt.imshow(img)
    plt.title(f"Age: {yb[i].item():.0f}")
    plt.axis("off")
plt.tight_layout()
plt.show()



In [ ]:
# Task 1: Write your model class here:
import torch.nn as nn

class AgeRegressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten
        return self.net(x)


In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0.0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset)


In [ ]:
# Task 3: Write your validation loop here:
@torch.no_grad()
def validate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = loss_fn(pred, yb)
        total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset)


In [ ]:
# Task 4: Define device, model, loss, optimizer:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

C, H, W = X_train.shape[1], X_train.shape[2], X_train.shape[3]
input_dim = C * H * W

model = AgeRegressor(input_dim).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
# Task 5: Start training for 20 epochs:
train_losses, val_losses = [], []

for epoch in range(20):
    tr_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    va_loss = validate(model, test_loader, loss_fn, device)

    train_losses.append(tr_loss)
    val_losses.append(va_loss)

    print(f"Epoch {epoch+1:02d} | Train Loss: {tr_loss:.4f} | Val Loss: {va_loss:.4f}")


In [ ]:
# Task 1: Write your code here:
plt.figure(figsize=(7,4))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Task 2 (Bonus): Write your code here:
@torch.no_grad()
def show_predictions(model, loader, device, n=6):
    model.eval()
    xb, yb = next(iter(loader))
    xb, yb = xb[:n].to(device), yb[:n].to(device)
    preds = model(xb).squeeze(1)

    plt.figure(figsize=(10, 4))
    for i in range(n):
        img = xb[i].cpu().permute(1, 2, 0).numpy()
        plt.subplot(2, 3, i+1)
        plt.imshow(img)
        plt.title(f"T:{yb[i].item():.0f} | P:{preds[i].item():.0f}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_predictions(model, test_loader, device, n=6)
